# County Reference Raw Data Quality

County FIPS reference used for joins and statewide FEMA expansion.

The audit queries the current `raw.*` ingestion tables directly; transformed tables are out of scope.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "data" / "housing_predict.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "housing_predict.duckdb"), read_only=True)
TABLES = ["counties"]
KEYS = {"counties":["fips"]}
YEAR_COLUMNS = {}
EXPECTED_YEARS = set(range(2010, 2025))
def qi(x): return '"' + x.replace('"', '""') + '"'
def columns(t):
    return con.execute("SELECT column_name FROM information_schema.columns WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [t]).df()["column_name"].tolist()
available = set(con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='raw'").df()["table_name"])
assert set(TABLES) <= available, f"Missing raw tables: {set(TABLES)-available}"


## Inventory, coverage, keys, and sampled missingness

In [2]:
inventory, key_checks, coverage, missing = [], [], [], []
for table in TABLES:
    cols = columns(table)
    n = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    inventory.append({"table": table, "rows": n, "columns": len(cols)})
    key = KEYS[table]
    key_sql = ", ".join(qi(c) for c in key)
    invalid = " OR ".join(f"{qi(c)} IS NULL OR trim(cast({qi(c)} AS VARCHAR))=''" for c in key)
    key_checks.append({
        "table": table, "candidate_key": " + ".join(key),
        "invalid_key_rows": con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}").fetchone()[0],
        "duplicate_key_groups": con.execute(f"SELECT count(*) FROM (SELECT {key_sql}, count(*) n FROM raw.{qi(table)} GROUP BY {key_sql} HAVING n>1)").fetchone()[0],
    })
    year_col = YEAR_COLUMNS.get(table)
    if year_col:
        years = set(con.execute(f"SELECT DISTINCT try_cast({qi(year_col)} AS INTEGER) FROM raw.{qi(table)} WHERE try_cast({qi(year_col)} AS INTEGER) IS NOT NULL").df().iloc[:,0])
        coverage.append({"table":table, "min_year":min(years), "max_year":max(years), "missing_years":sorted(EXPECTED_YEARS-years), "outside_scope":sorted(years-EXPECTED_YEARS)})
    # Identifier fields plus a bounded sample of wide value fields keeps the audit runnable.
    profile_cols = list(dict.fromkeys([*key, *(["NAME"] if "NAME" in cols else []), *cols[:40]]))
    for col in profile_cols:
        m = con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {qi(col)} IS NULL OR trim(cast({qi(col)} AS VARCHAR))=''").fetchone()[0]
        if m: missing.append({"table":table,"column":col,"missing_rows":m,"missing_pct":round(100*m/n,2)})
display(pd.DataFrame(inventory))
display(pd.DataFrame(coverage))
display(pd.DataFrame(key_checks))
display(pd.DataFrame(missing).sort_values("missing_pct",ascending=False).head(100) if missing else pd.DataFrame())


,table,rows,columns
0,counties,3218,9


""


,table,candidate_key,invalid_key_rows,duplicate_key_groups
0,counties,fips,0,0


,table,column,missing_rows,missing_pct
4,counties,csa_code,2068,64.26
5,counties,csa_name,2068,64.26
3,counties,msa_type,1411,43.85
1,counties,msa_code,1411,43.85
2,counties,msa_name,1411,43.85
0,counties,county_name,51,1.58


## Source-specific checks

In [3]:
display(con.execute("""
SELECT count(*) AS row_count, count(DISTINCT fips) AS distinct_fips,
 count(DISTINCT state) AS states_and_territories,
 count(*) FILTER (WHERE NOT regexp_matches(fips,'^[0-9]{5}$')) AS invalid_fips,
 count(*) FILTER (WHERE county_name IS NULL OR trim(county_name)='') AS missing_county_name,
 count(*) FILTER (WHERE msa_code IS NULL OR trim(msa_code)='') AS missing_msa
FROM raw.counties
""").df())
display(con.execute("""SELECT state,state_long,count(*) AS counties FROM raw.counties GROUP BY 1,2 ORDER BY 1""").df())

,row_count,distinct_fips,states_and_territories,invalid_fips,missing_county_name,missing_msa
0,3218,3218,51,0,51,1411


,state,state_long,counties
0,AK,Alaska,49
1,AL,Alabama,68
2,AR,Arkansas,76
3,AZ,Arizona,16
4,CA,California,59
5,CO,Colorado,65
6,CT,Connecticut,9
7,DC,District of Columbia,1
8,DC,istrict of Columbia,1
9,DE,Delaware,4


## Interpretation and limitations

This is a project crosswalk, not a vintage-aware Census geography history. County-equivalent changes may require temporal crosswalks. MSA/CSA fields are optional.

Displayed failures are investigation items. This notebook does not alter source data.

In [4]:
con.close()